# Baseline Experiments for the exchange rate dataset
#### Objective:The baseline model for the prediction project is established here
#### Metric:  sMAPE
#### Forecast-horizon: 24-step and 1-step

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import math


In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)


Using device: mps


In [3]:
df_train = pd.read_csv("../data/raw/train_data.csv")
df_val = pd.read_csv("../data/raw/validation_data.csv")

FEATURE_COLS = [c for c in df_train.columns if c != "timestamp_idx"]

train_data = df_train[FEATURE_COLS].values
val_data = df_val[FEATURE_COLS].values

print("Train shape:", train_data.shape)


Train shape: (5311, 9)


#### Creating Windows , Datasets and sMape function

In [4]:
def create_windows(data, seq_len, pred_len):
    X, Y = [], []
    for i in range(len(data) - seq_len - pred_len + 1):
        X.append(data[i:i+seq_len])
        Y.append(data[i+seq_len:i+seq_len+pred_len])
    return np.array(X), np.array(Y)


In [5]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


In [6]:
def smape(y_true, y_pred):
    eps = 1e-8
    return torch.mean(
        2 * torch.abs(y_true - y_pred) /
        (torch.abs(y_true) + torch.abs(y_pred) + eps)
    )


In [7]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


## Experiment 1

#### Here a full Transformer model is used to do raw level prediction 

In [50]:
SEQ_LEN = 96
PRED_LEN = 24
input_dim = train_data.shape[1]
D_MODEL = 64
N_HEADS = 4
ENC_LAYERS = 2
DEC_LAYERS = 2
DROPOUT = 0.1


In [51]:
class EncoderDecoderTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, D_MODEL)
        self.pos_enc = PositionalEncoding(D_MODEL)

        self.transformer = nn.Transformer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            num_encoder_layers=ENC_LAYERS,
            num_decoder_layers=DEC_LAYERS,
            dropout=DROPOUT,
            batch_first=True
        )

        self.output_proj = nn.Linear(D_MODEL, input_dim)

    def forward(self, src, tgt):
        src = self.input_proj(src)
        tgt = self.input_proj(tgt)

        src = self.pos_enc(src)
        tgt = self.pos_enc(tgt)

        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            tgt.size(1)
        ).to(src.device)

        out = self.transformer(src, tgt, tgt_mask=tgt_mask)
        out = self.output_proj(out)

        return out[:, -PRED_LEN:, :]


In [52]:
X_train, Y_train = create_windows(train_data, SEQ_LEN, PRED_LEN)
X_val, Y_val     = create_windows(val_data, SEQ_LEN, PRED_LEN)


In [53]:
def train_model(model, train_loader, val_loader, residual=False):

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()

    best_val = float("inf")

    for epoch in range(10):
        model.train()
        train_loss = 0

        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            optimizer.zero_grad()
            dec_input = torch.zeros(
            Y_batch.shape[0],
            PRED_LEN,
            input_dim
            ).to(device)

            output = model(X_batch, dec_input)

            loss = criterion(output, Y_batch)
            loss.backward()
            # Create decoder input
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()

        # Validation        
        model.eval()
        val_smape = 0

        with torch.no_grad():
            for X_batch, Y_batch in val_loader:

                X_batch = X_batch.to(device)
                Y_batch = Y_batch.to(device)
                # Create decoder input
                dec_input = torch.zeros(
                 Y_batch.shape[0],
                 PRED_LEN,
                 input_dim
                    ).to(device)

                output = model(X_batch, dec_input)

            

                if residual:
                    last_value = X_batch[:, -1:, :]
                    final_pred = last_value + output
                    Y_original = last_value + Y_batch
                else:
                    final_pred = output
                    Y_original = Y_batch

                val_smape += smape(Y_original, final_pred).item()

        val_smape /= len(val_loader)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val sMAPE: {val_smape:.4f}")

    return val_smape


In [54]:
train_dataset = TimeSeriesDataset(X_train, Y_train)
val_dataset   = TimeSeriesDataset(X_val, Y_val)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)


In [55]:
# Instantiate model
model = EncoderDecoderTransformer().to(device)

# Call training loop
final_val_smape_1= train_model(
    model,
    train_loader,
    val_loader,
    residual=False
)

print("Final Validation sMAPE:", final_val_smape_1)


Epoch 1 | Train Loss: 1060939.1741 | Val sMAPE: 1.8598
Epoch 2 | Train Loss: 1059250.5694 | Val sMAPE: 1.9672
Epoch 3 | Train Loss: 1057314.7440 | Val sMAPE: 1.8910
Epoch 4 | Train Loss: 1055176.4184 | Val sMAPE: 1.8004
Epoch 5 | Train Loss: 1052857.5464 | Val sMAPE: 1.7643
Epoch 6 | Train Loss: 1050367.3011 | Val sMAPE: 1.7551
Epoch 7 | Train Loss: 1047710.1976 | Val sMAPE: 1.7498
Epoch 8 | Train Loss: 1044888.7391 | Val sMAPE: 1.7441
Epoch 9 | Train Loss: 1041904.7365 | Val sMAPE: 1.7383
Epoch 10 | Train Loss: 1038759.7583 | Val sMAPE: 1.7301
Final Validation sMAPE: 1.7301424950361253


## Experiment 2

In [56]:
SEQ_LEN = 96
PRED_LEN = 24
input_dim = train_data.shape[1]
D_MODEL = 64
N_HEADS = 4
ENC_LAYERS = 2
DEC_LAYERS = 2
DROPOUT = 0.1


In [57]:
class EncoderOnlyTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, D_MODEL)
        self.pos_enc = PositionalEncoding(D_MODEL)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            dropout=DROPOUT,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=ENC_LAYERS
        )

        self.output_proj = nn.Linear(D_MODEL, PRED_LEN * input_dim)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_enc(x)
        x = self.encoder(x)

        x = x[:, -1, :]
        out = self.output_proj(x)
        out = out.view(-1, PRED_LEN, input_dim)

        return out


In [58]:
def train_model(model, train_loader, val_loader, residual=False):

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()

    best_val = float("inf")

    for epoch in range(10):
        model.train()
        train_loss = 0

        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            optimizer.zero_grad()
            output = model(X_batch)

            loss = criterion(output, Y_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()

        # Validation
        model.eval()
        val_smape = 0

        with torch.no_grad():
            for X_batch, Y_batch in val_loader:

                X_batch = X_batch.to(device)
                Y_batch = Y_batch.to(device)

                output = model(X_batch)

                if residual:
                    last_value = X_batch[:, -1:, :]
                    final_pred = last_value + output
                    Y_original = last_value + Y_batch
                else:
                    final_pred = output
                    Y_original = Y_batch

                val_smape += smape(Y_original, final_pred).item()

        val_smape /= len(val_loader)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val sMAPE: {val_smape:.4f}")

    return val_smape


In [ ]:
# Step 1 — Create windows (RAW targets, no residual)
X_train, Y_train = create_windows(train_data, SEQ_LEN, PRED_LEN)
X_val, Y_val     = create_windows(val_data, SEQ_LEN, PRED_LEN)

print("Train windows:", X_train.shape)
print("Val windows:", X_val.shape)

# Step 2 — Create datasets
train_dataset = TimeSeriesDataset(X_train, Y_train)
val_dataset   = TimeSeriesDataset(X_val, Y_val)

# Step 3 — Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Step 4 — Instantiate model
model = EncoderOnlyTransformer().to(device)

# Step 5 — CALL training loop
final_val_smape_2 = train_model(
    model,
    train_loader,
    val_loader,
      # IMPORTANT → raw experiment
)

print("Final Validation sMAPE (Experiment 2):", final_val_smape_2)


Train windows: (5192, 96, 9)
Val windows: (640, 96, 9)
Epoch 1 | Train Loss: 1063210.7377 | Val sMAPE: 1.8000
Epoch 2 | Train Loss: 1061589.3237 | Val sMAPE: 1.8384
Epoch 3 | Train Loss: 1059821.4593 | Val sMAPE: 1.8027
Epoch 4 | Train Loss: 1057921.3301 | Val sMAPE: 1.7764
Epoch 5 | Train Loss: 1055886.4574 | Val sMAPE: 1.7680
Epoch 6 | Train Loss: 1053715.1523 | Val sMAPE: 1.7661
Epoch 7 | Train Loss: 1051404.9455 | Val sMAPE: 1.7628
Epoch 8 | Train Loss: 1048953.0302 | Val sMAPE: 1.7577
Epoch 9 | Train Loss: 1046357.9435 | Val sMAPE: 1.7512
Epoch 10 | Train Loss: 1043617.3423 | Val sMAPE: 1.7444
Final Validation sMAPE (Experiment 2): 1.7443570435047149


## Experiment 3

In [60]:
SEQ_LEN = 96
PRED_LEN = 24
input_dim = train_data.shape[1]
D_MODEL = 64
N_HEADS = 4
ENC_LAYERS = 2
DEC_LAYERS = 2
DROPOUT = 0.1


In [61]:
X_train, Y_train = create_windows(train_data, SEQ_LEN, 24)
X_val, Y_val = create_windows(val_data, SEQ_LEN, 24)

Y_train_res = Y_train - X_train[:, -1:, :]
Y_val_res   = Y_val - X_val[:, -1:, :]


In [62]:
class EncoderOnlyTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, D_MODEL)
        self.pos_enc = PositionalEncoding(D_MODEL)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            dropout=DROPOUT,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=ENC_LAYERS
        )

        self.output_proj = nn.Linear(D_MODEL, PRED_LEN * input_dim)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_enc(x)
        x = self.encoder(x)

        x = x[:, -1, :]
        out = self.output_proj(x)
        out = out.view(-1, PRED_LEN, input_dim)

        return out


In [63]:
def train_model(model, train_loader, val_loader, residual=False):

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()

    best_val = float("inf")

    for epoch in range(10):
        model.train()
        train_loss = 0

        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            optimizer.zero_grad()
            output = model(X_batch)

            loss = criterion(output, Y_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()

        # Validation
        model.eval()
        val_smape = 0

        with torch.no_grad():
            for X_batch, Y_batch_res in val_loader:

                X_batch = X_batch.to(device)
                Y_batch_res = Y_batch_res.to(device)

                residual_pred = model(X_batch)

                last_value = X_batch[:, -1:, :]
                final_pred = last_value + residual_pred
                Y_original = last_value + Y_batch_res

                val_smape += smape(Y_original, final_pred).item()

        val_smape /= len(val_loader)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val sMAPE: {val_smape:.4f}")

    return val_smape


In [64]:
# Create datasets
train_dataset = TimeSeriesDataset(X_train, Y_train_res)
val_dataset   = TimeSeriesDataset(X_val, Y_val_res)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Instantiate model
model = EncoderOnlyTransformer().to(device)

# Call training loop
final_val_smape_3 = train_model(
    model,
    train_loader,
    val_loader,
    residual=True
)

print("Final Validation sMAPE:", final_val_smape_3)


Epoch 1 | Train Loss: 17.1757 | Val sMAPE: 0.0766
Epoch 2 | Train Loss: 11.5876 | Val sMAPE: 0.0740
Epoch 3 | Train Loss: 6.9577 | Val sMAPE: 0.0732
Epoch 4 | Train Loss: 3.5206 | Val sMAPE: 0.0730
Epoch 5 | Train Loss: 1.2964 | Val sMAPE: 0.0730
Epoch 6 | Train Loss: 0.2476 | Val sMAPE: 0.0774
Epoch 7 | Train Loss: 0.0465 | Val sMAPE: 0.0743
Epoch 8 | Train Loss: 0.0301 | Val sMAPE: 0.0740
Epoch 9 | Train Loss: 0.0290 | Val sMAPE: 0.0742
Epoch 10 | Train Loss: 0.0288 | Val sMAPE: 0.0742
Final Validation sMAPE: 0.07421849044039845


## Experiment 4

In [65]:
SEQ_LEN = 96
PRED_LEN = 1
input_dim = train_data.shape[1]
D_MODEL = 64
N_HEADS = 4
ENC_LAYERS = 2
DEC_LAYERS = 2
DROPOUT = 0.1


In [66]:
X_train, Y_train = create_windows(train_data, SEQ_LEN, 1)
X_val, Y_val = create_windows(val_data, SEQ_LEN, 1)

Y_train_res = Y_train - X_train[:, -1:, :]
Y_val_res   = Y_val - X_val[:, -1:, :]


In [67]:
class EncoderOnlyTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, D_MODEL)
        self.pos_enc = PositionalEncoding(D_MODEL)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            dropout=DROPOUT,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=ENC_LAYERS
        )

        self.output_proj = nn.Linear(D_MODEL, PRED_LEN * input_dim)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_enc(x)
        x = self.encoder(x)

        x = x[:, -1, :]
        out = self.output_proj(x)
        out = out.view(-1, PRED_LEN, input_dim)

        return out


In [68]:
def train_model(model, train_loader, val_loader, residual=False):

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()

    best_val = float("inf")

    for epoch in range(10):
        model.train()
        train_loss = 0

        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            optimizer.zero_grad()
            output = model(X_batch)

            loss = criterion(output, Y_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()

        # Validation
        model.eval()
        val_smape = 0

        with torch.no_grad():
            for X_batch, Y_batch_res in val_loader:

                X_batch = X_batch.to(device)
                Y_batch_res = Y_batch_res.to(device)

                residual_pred = model(X_batch)

                last_value = X_batch[:, -1:, :]
                final_pred = last_value + residual_pred
                Y_original = last_value + Y_batch_res

                val_smape += smape(Y_original, final_pred).item()

        val_smape /= len(val_loader)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val sMAPE: {val_smape:.4f}")

    return val_smape


In [69]:
print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)


X_train shape: (5215, 96, 9)
Y_train shape: (5215, 1, 9)


In [ ]:
#  Step 1 — Create datasets (RESIDUAL targets)
train_dataset = TimeSeriesDataset(X_train, Y_train_res)
val_dataset   = TimeSeriesDataset(X_val, Y_val_res)

#  Step 2 — Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

#  Step 3 — Instantiate model
model = EncoderOnlyTransformer().to(device)

# Step 4 — Call training loop
final_val_smape_4= train_model(
    model,
    train_loader,
    val_loader,
    residual=True   # IMPORTANT → residual reconstruction used
)

print("Final Validation sMAPE (Experiment 4 - 1 Step):", final_val_smape_4)


Train batches: 326
Val batches: 42
Epoch 1 | Train Loss: 0.0110 | Val sMAPE: 0.0265
Epoch 2 | Train Loss: 0.0053 | Val sMAPE: 0.0262
Epoch 3 | Train Loss: 0.0043 | Val sMAPE: 0.0251
Epoch 4 | Train Loss: 0.0038 | Val sMAPE: 0.0230
Epoch 5 | Train Loss: 0.0035 | Val sMAPE: 0.0219
Epoch 6 | Train Loss: 0.0034 | Val sMAPE: 0.0216
Epoch 7 | Train Loss: 0.0032 | Val sMAPE: 0.0242
Epoch 8 | Train Loss: 0.0031 | Val sMAPE: 0.0225
Epoch 9 | Train Loss: 0.0031 | Val sMAPE: 0.0223
Epoch 10 | Train Loss: 0.0030 | Val sMAPE: 0.0231
Final Validation sMAPE (Experiment 4 - 1 Step): 0.023064252233044022


## Naive Baselines

In [43]:
def compute_naive(X, Y):
    X = torch.tensor(X, dtype=torch.float32).to(device)
    Y = torch.tensor(Y, dtype=torch.float32).to(device)

    last_value = X[:, -1:, :]
    naive_pred = last_value.repeat(1, Y.shape[1], 1)

    return smape(Y, naive_pred).item()


For Experiments 1,2,3 :


In [44]:
# Create validation windows
X_val, Y_val = create_windows(val_data, SEQ_LEN, PRED_LEN)

# Compute naive baseline
naive_smape = compute_naive(X_val, Y_val)

print("Naive Validation sMAPE (24-step):", naive_smape)


Naive Validation sMAPE (24-step): 0.02147861011326313


For. Experiment 4:


In [46]:
# 1-step validation windows
X_val_1, Y_val_1 = create_windows(val_data, SEQ_LEN, 1)

naive_smape_1 = compute_naive(X_val_1, Y_val_1)

print("Naive Validation sMAPE (1-step):", naive_smape_1)


Naive Validation sMAPE (1-step): 0.02147861011326313


In [71]:
results={}

results["Encoder-Decoder Raw 24"] = {
    "model_smape": final_val_smape_1,
    "naive_smape": naive_smape
}


results["Encoder Raw 24"] = {
    "model_smape": final_val_smape_2,
    "naive_smape": naive_smape
}
results["Encoder Residual 24"] = {
    "model_smape": final_val_smape_3,
    "naive_smape": naive_smape
}
results["Encoder Residual 1"] = {
    "model_smape": final_val_smape_4,
    "naive_smape": naive_smape_1
}




In [72]:
comparison=pd.DataFrame(results).T
comparison["beats_naive"]=comparison["model_smape"]< comparison["naive_smape"]

In [73]:
comparison

,model_smape,naive_smape,beats_naive
Encoder-Decoder Raw 24,1.730142,0.021479,False
Encoder Raw 24,1.744357,0.021479,False
Encoder Residual 24,0.074218,0.021479,False
Encoder Residual 1,0.023064,0.021479,False


##### None of the Transformer-based models outperform the naive persistence baseline.
The 1-step residual model converges to naive performance, confirming strong persistence behavior in the dataset.